In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')


In [ ]:


import os, glob, numpy as np, librosa
from scipy.signal import butter, lfilter
from tqdm import tqdm


ICBHI_WAV_DIR = "/content/drive/MyDrive/ICBHI_2017/audio_and_txt_files"
ICBHI_OUT_DIR = "/content/drive/MyDrive/ICBHI_2017/feature"
os.makedirs(ICBHI_OUT_DIR, exist_ok=True)


SR = 4000
N_FFT = 256
HOP = 64             # ~16 ms
N_MELS = 64
FMAX = 2000
WIN_FRAMES = 64      # ~1.024 s
STRIDE_FRAMES = 32   # ~0.512 s
SAVE_FLOAT16 = True  # halves disk usage

def high_pass_filter(x, sr, cutoff=80, order=8):
    nyq = 0.5*sr
    b, a = butter(order, cutoff/nyq, btype='high', analog=False)
    return lfilter(b, a, x)

def compute_logmel(y):
    M = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
        n_mels=N_MELS, fmax=FMAX, power=2.0
    )
    S = librosa.power_to_db(M, ref=np.max).T.astype(np.float32)  # (T, 64)
    mu, sd = S.mean(axis=0), S.std(axis=0) + 1e-6                # per-clip z-score
    return ((S - mu) / sd).astype(np.float32)

def wav_to_windows(wav_path):
    y, sr0 = librosa.load(wav_path, sr=None, mono=True)
    if sr0 != SR:
        y = librosa.resample(y=y, orig_sr=sr0, target_sr=SR)
    y = high_pass_filter(y, SR)
    mel = compute_logmel(y)                      # (T, 64)
    T, F = mel.shape
    if T < WIN_FRAMES:
        return None, None
    spans = [(s, s + WIN_FRAMES) for s in range(0, T - WIN_FRAMES + 1, STRIDE_FRAMES)]
    L = len(spans)
    X = np.empty((L, 1, WIN_FRAMES, F), dtype=np.float32)
    for i, (a, b) in enumerate(spans):
        X[i, 0] = mel[a:b, :]
    return X, np.array(spans, dtype=np.int32)

def precompute_icbhi(wav_root, out_root):
    wavs = sorted(
        glob.glob(os.path.join(wav_root, "**", "*.wav"), recursive=True) +
        glob.glob(os.path.join(wav_root, "**", "*.WAV"), recursive=True)
    )
    print(f"[INFO] Found {len(wavs)} wav files under {wav_root}")
    saved = skipped = errors = 0
    for w in tqdm(wavs, desc="Precomputing windows"):
        stem = os.path.splitext(os.path.basename(w))[0]
        X_path = os.path.join(out_root, f"{stem}_windows.npy")
        S_path = os.path.join(out_root, f"{stem}_winspans.npy")
        if os.path.exists(X_path) and os.path.exists(S_path):
            continue  # resume-friendly
        try:
            X, spans = wav_to_windows(w)
            if X is None:
                skipped += 1
                continue
            if SAVE_FLOAT16:
                np.save(X_path, X.astype(np.float16))
            else:
                np.save(X_path, X)
            np.save(S_path, spans)
            saved += 1
        except Exception as e:
            errors += 1
            print(f"[ERR] {stem}: {e}")
    print(f"[SUMMARY] saved={saved} | skipped_short={skipped} | errors={errors} | total_wavs={len(wavs)}")


precompute_icbhi(ICBHI_WAV_DIR, ICBHI_OUT_DIR)

import random
files = [f for f in os.listdir(ICBHI_OUT_DIR) if f.endswith("_windows.npy")]
print("NPY files:", len(files))
for f in random.sample(files, min(3, len(files))):
    stem = f.replace("_windows.npy","")
    X = np.load(os.path.join(ICBHI_OUT_DIR, f))
    S = np.load(os.path.join(ICBHI_OUT_DIR, stem + "_winspans.npy"))
    print(stem, "->", X.shape, "| spans:", S.shape)

[INFO] Found 920 wav files under /content/drive/MyDrive/ICBHI_2017/audio_and_txt_files


Precomputing windows: 100%|██████████| 920/920 [00:05<00:00, 154.61it/s]


[SUMMARY] saved=0 | skipped_short=0 | errors=0 | total_wavs=920
NPY files: 920
133_2p4_Tc_mc_AKGC417L -> (38, 1, 64, 64) | spans: (38, 2)
158_1p3_Ar_mc_AKGC417L -> (38, 1, 64, 64) | spans: (38, 2)
205_3b4_Al_mc_AKGC417L -> (38, 1, 64, 64) | spans: (38, 2)


In [ ]:


import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm

FEATURES_DIR = "/content/drive/MyDrive/ICBHI_2017/feature"
ENC_SAVE     = "/content/drive/MyDrive/ICBHI_2017/pretrained_windowcnn_icbhi_ssl.pt"

def time_mask(mel, T=64, max_t=10):
    t = np.random.randint(0, max_t+1)
    if t==0: return mel
    t0 = np.random.randint(0, T - t + 1)
    mel[t0:t0+t,:] = 0.0
    return mel

def freq_mask(mel, F=64, max_f=8):
    f = np.random.randint(0, max_f+1)
    if f==0: return mel
    f0 = np.random.randint(0, F - f + 1)
    mel[:,f0:f0+f] = 0.0
    return mel

def random_time_shift(mel, max_shift=4):
    s = np.random.randint(-max_shift, max_shift+1)
    if s==0: return mel
    if s>0:  return np.concatenate([mel[s:], np.zeros_like(mel[:s])], 0)
    s = -s;  return np.concatenate([np.zeros_like(mel[:s]), mel[:-s]], 0)

def add_gaussian(mel, std=0.015):
    return mel + np.random.normal(0.0, std, mel.shape).astype(np.float32)

def augment_view(win):
    x = win.astype(np.float32).copy()   # (1,64,64)
    x0 = x[0]
    x0 = random_time_shift(x0, 4)
    if np.random.rand()<0.9: x0 = time_mask(x0, 64, 10)
    if np.random.rand()<0.9: x0 = freq_mask(x0, 64, 8)
    if np.random.rand()<0.5: x0 = add_gaussian(x0, 0.015)
    x[0] = x0
    return x


class WindowsSimCLR(Dataset):
    def __init__(self, features_dir):
        self.index = []
        for name in os.listdir(features_dir):
            if not name.endswith("_windows.npy"): continue
            p = os.path.join(features_dir, name)
            try:
                L = np.load(p, mmap_mode='r').shape[0]
                if L>0: self.index.extend([(p, i) for i in range(L)])
            except Exception:
                pass
        if not self.index:
            raise RuntimeError("No windows found.")
    def __len__(self): return len(self.index)
    def __getitem__(self, idx):
        p, i = self.index[idx]
        w = np.load(p, mmap_mode='r')[i]       # (1,64,64) float16/32
        v1 = torch.from_numpy(augment_view(w)).float()
        v2 = torch.from_numpy(augment_view(w)).float()
        return v1, v2


class WindowCNN(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), # 64→32
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.proj = nn.Linear(128, 128)
    def forward(self, x):
        h = self.net(x).squeeze(-1).squeeze(-1)
        return self.proj(h)

class SimCLR(nn.Module):
    def __init__(self, emb_dim=128, proj_dim=128):
        super().__init__()
        self.encoder = WindowCNN(emb_dim)
        self.projector = nn.Sequential(
            nn.Linear(emb_dim, emb_dim), nn.ReLU(inplace=True),
            nn.Linear(emb_dim, proj_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return nn.functional.normalize(self.projector(z), dim=-1)

def nt_xent(z1, z2, temperature=0.15):
    B = z1.size(0)
    z = torch.cat([z1, z2], 0)                 # (2B,D)
    sim = (z @ z.t()) / temperature            # (2B,2B)

    eye = torch.eye(2*B, device=sim.device, dtype=torch.bool)
    sim.masked_fill_(eye, torch.finfo(sim.dtype).min)
    targets = torch.cat([torch.arange(B,2*B), torch.arange(0,B)], 0).to(z.device)
    return nn.CrossEntropyLoss()(sim, targets)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH, EPOCHS = 128, 30
LR, WD, TEMP = 1e-3, 1e-4, 0.15
ACCUM = 1

torch.backends.cudnn.benchmark = True
ds = WindowsSimCLR(FEATURES_DIR)
loader = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=4,
                    pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)
model = SimCLR(emb_dim=128, proj_dim=128).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = GradScaler()

print(f"[INFO] windows: {len(ds)}  | batches/epoch≈{max(1,len(ds)//BATCH)}")
for ep in range(1, EPOCHS+1):
    model.train(); run = 0.0
    pbar = tqdm(loader, desc=f"SimCLR {ep}/{EPOCHS}")
    for step, (v1, v2) in enumerate(pbar):
        v1,v2 = v1.to(device,non_blocking=True), v2.to(device,non_blocking=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            p1 = model(v1); p2 = model(v2)
            loss = nt_xent(p1,p2,temperature=TEMP) / ACCUM
        scaler.scale(loss).backward()
        if (step+1)%ACCUM==0:
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
        run += loss.item()*ACCUM
        pbar.set_postfix(loss=f"{(run/(step+1)):.4f}")
    print(f"  ↳ avg loss: {run/max(1,len(loader)):.4f}")

os.makedirs(os.path.dirname(ENC_SAVE), exist_ok=True)
torch.save(model.encoder.state_dict(), ENC_SAVE)
print("Saved pretrained encoder →", ENC_SAVE)


[INFO] windows: 37602  | batches/epoch≈293


SimCLR 1/30: 100%|██████████| 293/293 [02:08<00:00,  2.27it/s, loss=1.4348]


  ↳ avg loss: 1.4348


SimCLR 2/30: 100%|██████████| 293/293 [01:59<00:00,  2.44it/s, loss=0.9056]


  ↳ avg loss: 0.9056


...

  ↳ avg loss: 0.5855


SimCLR 29/30: 100%|██████████| 293/293 [02:00<00:00,  2.43it/s, loss=0.5845]


  ↳ avg loss: 0.5845


SimCLR 30/30: 100%|██████████| 293/293 [02:01<00:00,  2.42it/s, loss=0.5807]


  ↳ avg loss: 0.5807
Saved pretrained encoder → /content/drive/MyDrive/ICBHI_2017/pretrained_windowcnn_icbhi_ssl.pt


In [ ]:

import os, glob, re, numpy as np, pandas as pd, librosa
from scipy.signal import butter, lfilter
from tqdm import tqdm

HF_TRAIN_DIR = "/content/drive/MyDrive/HF_Lung_V1/train"
HF_TEST_DIR  = "/content/drive/MyDrive/HF_Lung_V1/test"
OUT_TRAIN    = "/content/drive/MyDrive/HF_Lung_V1/train/feature"
OUT_TEST     = "/content/drive/MyDrive/HF_Lung_V1/test/feature"
os.makedirs(OUT_TRAIN, exist_ok=True); os.makedirs(OUT_TEST, exist_ok=True)

SR=4000; N_FFT=256; HOP=64; N_MELS=64; FMAX=2000
WIN_FRAMES=64; STRIDE_FRAMES=32
SAVE_FLOAT16=True
SAVE_TIMELINE=False
MIN_OVERLAP_FRAC=0.0

ADVENT=['wheeze','crackle','rhonchi','stridor']
TOKEN_MAP={'w':'wheeze','wheeze':'wheeze','d':'crackle','crackle':'crackle','crackles':'crackle',
           'r':'rhonchi','rhonchi':'rhonchi','s':'stridor','stridor':'stridor',
           'i':'inhalation','inhalation':'inhalation','e':'exhalation','exhalation':'exhalation'}

def high_pass_filter(x, sr, cutoff=80, order=8):
    nyq=0.5*sr; b,a=butter(order, cutoff/nyq, btype='high', analog=False)
    return lfilter(b,a,x)

def compute_logmel(y):
    M = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                       n_mels=N_MELS, fmax=FMAX, power=2.0)
    S = librosa.power_to_db(M, ref=np.max).T.astype(np.float32)  # (T,64)
    mu, sd = S.mean(axis=0), S.std(axis=0) + 1e-6
    return ((S - mu)/sd).astype(np.float32)

def extract_windows_for_wav(wav_path):
    y, sr0 = librosa.load(wav_path, sr=None, mono=True)
    if sr0 != SR:
        y = librosa.resample(y=y, orig_sr=sr0, target_sr=SR)
    y = high_pass_filter(y, SR)
    mel = compute_logmel(y)                      # (T,64)
    T, F = mel.shape
    if T < WIN_FRAMES:
        return None, None
    spans=[]; s=0
    while s + WIN_FRAMES <= T:
        spans.append((s, s + WIN_FRAMES))
        s += STRIDE_FRAMES
    L=len(spans)
    X = np.empty((L,1,WIN_FRAMES,F), dtype=np.float32)
    for i,(a,b) in enumerate(spans):
        X[i,0] = mel[a:b,:]
    return X, np.array(spans, dtype=np.int32)

def ts_to_sec(t):
    t=t.strip()
    m=re.match(r"^(?:(\d+):)?(\d+):(\d+(?:\.\d+)?)$", t)
    if m:
        h=int(m.group(1) or 0); mnt=int(m.group(2)); s=float(m.group(3))
        return h*3600 + mnt*60 + s
    try: return float(t)
    except: return None

def parse_hf_label_file(txt_path):
    ints={c:[] for c in ADVENT}
    if not (txt_path and os.path.exists(txt_path)): return ints
    with open(txt_path,'r') as f:
        for line in f:
            parts=re.split(r'\s+', line.strip())
            if len(parts)<3: continue
            lbl=TOKEN_MAP.get(parts[0].lower())
            if lbl not in ADVENT: continue
            s=ts_to_sec(parts[1]); e=ts_to_sec(parts[2])
            if s is None or e is None or e<=s: continue
            ints[lbl].append((s,e))
    return ints

def label_windows(spans, intervals):
    f2s=lambda f:(f*HOP)/float(SR)
    L=len(spans); Y=np.zeros((L,len(ADVENT)), dtype=np.float32)
    for i,(a,b) in enumerate(spans):
        ws,we=f2s(int(a)), f2s(int(b)); wdur=we-ws
        for ci,cls in enumerate(ADVENT):
            for (s,e) in intervals.get(cls,[]):
                ov=max(0.0, min(we,e)-max(ws,s))
                if ov >= MIN_OVERLAP_FRAC*wdur:
                    Y[i,ci]=1.0; break
    return Y

def timeline_df(spans, Y):
    f2s=lambda f:(f*HOP)/float(SR)
    rows=[]
    for i,(a,b) in enumerate(spans):
        row={"win_idx":i,"start_s":f2s(int(a)),"end_s":f2s(int(b))}
        for ci,cls in enumerate(ADVENT): row[cls]=int(Y[i,ci])
        rows.append(row)
    return pd.DataFrame(rows)

def find_label_file(folder, stem):
    for cand in (f"{stem}_label.txt", f"{stem}.txt", f"{stem}_labels.txt"):
        p=os.path.join(folder, cand)
        if os.path.exists(p): return p
    return None

def process_split(audio_dir, out_dir):
    wavs=sorted(glob.glob(os.path.join(audio_dir,"*.wav")) + glob.glob(os.path.join(audio_dir,"*.WAV")))
    print(f"[INFO] {audio_dir}: {len(wavs)} wav files")
    saved=missing_labels=errs=skipped_short=0
    for w in tqdm(wavs, desc=f"HF_Lung_V1: {os.path.basename(audio_dir)}"):
        stem=os.path.splitext(os.path.basename(w))[0]
        Xp=os.path.join(out_dir, f"{stem}_windows.npy")
        Yp=os.path.join(out_dir, f"{stem}_winlabels.npy")
        Sp=os.path.join(out_dir, f"{stem}_winspans.npy")
        TL=os.path.join(out_dir, f"{stem}_timeline.csv")
        if os.path.exists(Xp) and os.path.exists(Yp) and os.path.exists(Sp): continue
        try:
            X, spans = extract_windows_for_wav(w)
            if X is None:      # <— skip ultra-short
                skipped_short += 1; continue
            lbl_path = find_label_file(audio_dir, stem)
            intervals = parse_hf_label_file(lbl_path) if lbl_path else {c:[] for c in ADVENT}
            if not lbl_path: missing_labels += 1
            Y = label_windows(spans, intervals)
            np.save(Xp, X.astype(np.float16) if SAVE_FLOAT16 else X)
            np.save(Yp, Y.astype(np.float32))
            np.save(Sp, spans.astype(np.int32))
            if SAVE_TIMELINE:
                tl=timeline_df(spans, Y); tl.insert(0,"filename", os.path.basename(w)); tl.to_csv(TL, index=False)
            saved += 1
        except Exception as e:
            errs += 1; print(f"[ERR] {stem}: {e}")
    print(f"[SUMMARY {os.path.basename(audio_dir)}] saved={saved} | skipped_short={skipped_short} | missing_label={missing_labels} | errors={errs} | total_wavs={len(wavs)}")

process_split(HF_TRAIN_DIR, OUT_TRAIN)



[INFO] /content/drive/MyDrive/HF_Lung_V1/train: 7809 wav files


HF_Lung_V1: train: 100%|██████████| 7809/7809 [08:59<00:00, 14.49it/s]

[SUMMARY train] saved=7809 | skipped_short=0 | missing_label=0 | errors=0 | total_wavs=7809


In [ ]:
process_split(HF_TEST_DIR,  OUT_TEST)

[INFO] /content/drive/MyDrive/HF_Lung_V1/test: 1956 wav files


HF_Lung_V1: test: 100%|██████████| 1956/1956 [01:58<00:00, 16.48it/s]

[SUMMARY test] saved=1956 | skipped_short=0 | missing_label=0 | errors=0 | total_wavs=1956


In [ ]:


!pip install -q scikit-learn==1.5.1 tqdm==4.66.4

import os, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score
from tqdm import tqdm


TRAIN_FEATURES_DIR = "/content/drive/MyDrive/HF_Lung_V1/train/feature"
TEST_FEATURES_DIR  = "/content/drive/MyDrive/HF_Lung_V1/test/feature"
SSL_ENCODER_PATH   = "/content/drive/MyDrive/ICBHI_2017/pretrained_windowcnn_icbhi_ssl.pt"  # optional
BEST_MODEL_PATH    = "/content/drive/MyDrive/Conformer/hf_lung_model_best.pt"


SEED = 1337
BATCH_CLIPS = 4
EPOCHS = 50
FREEZE_EPOCHS = 3
BASE_LR_HEAD = 5e-4
BASE_LR_CNN  = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0
VAL_SPLIT = 0.1
NUM_WORKERS = 2


ASL_GAMMA_POS = 0.0
ASL_GAMMA_NEG = 4.0
ASL_CLIP = 0.05


AUG_TRAIN = True
AUG_PROB  = 0.5          # per-window chance to augment
FILTER_GAIN_DB = 4.0     # FilterAugment strength


def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def list_stems(feat_dir):
    stems = []
    for f in os.listdir(feat_dir):
        if f.endswith("_windows.npy"):
            stems.append(os.path.join(feat_dir, f[:-12]))
    stems.sort()
    return stems

def split_train_val(stems, val_frac=VAL_SPLIT, seed=SEED):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(stems)); rng.shuffle(idx)
    n_val = max(1, int(len(stems)*val_frac))
    return [stems[i] for i in idx[n_val:]], [stems[i] for i in idx[:n_val]]


def _time_shift_(x2d, max_shift=2):

    s = int(torch.randint(-max_shift, max_shift+1, (1,)).item())
    if s == 0: return x2d
    x2d_copy = x2d.clone()
    if s > 0:
        x2d[s:] = x2d_copy[:-s]
        x2d[:s] = 0
    else:
        s = -s
        x2d[:-s] = x2d_copy[s:]
        x2d[-s:] = 0
    return x2d

def _time_mask_(x2d, max_t=10):
    if torch.rand(1).item() < 0.7:
        t = int(torch.randint(1, max_t+1, (1,)).item())
        t0 = int(torch.randint(0, 64 - t + 1, (1,)).item())
        x2d[t0:t0+t, :] = 0
    return x2d

def _freq_mask_(x2d, max_f=8):
    if torch.rand(1).item() < 0.7:
        f = int(torch.randint(1, max_f+1, (1,)).item())
        f0 = int(torch.randint(0, 64 - f + 1, (1,)).item())
        x2d[:, f0:f0+f] = 0
    return x2d

def _filter_augment_(x2d, max_gain_db=FILTER_GAIN_DB):

    if torch.rand(1).item() < 0.7:
        f0 = int(torch.randint(0, 48, (1,)).item())
        width = int(torch.randint(8, 24, (1,)).item())
        f1 = min(64, f0 + width)
        gain = (torch.rand(1).item()*2 - 1) * max_gain_db  # [-g, +g] dB-ish
        x2d[:, f0:f1] = x2d[:, f0:f1] + (gain / 20.0)
    return x2d

def aug_window(win):
    # win: (1,64,64) float32 tensor (log-mel z-scored)
    x = win.clone()
    x0 = x[0]
    x0 = _time_shift_(x0, max_shift=2)
    x0 = _time_mask_(x0, max_t=10)
    x0 = _freq_mask_(x0, max_f=8)
    x0 = _filter_augment_(x0, max_gain_db=FILTER_GAIN_DB)
    x[0] = x0
    return x

class HFLungWindows(Dataset):
    def __init__(self, stems, with_labels=True, augment=False, aug_prob=AUG_PROB):
        self.stems = stems
        self.with_labels = with_labels
        self.augment = augment
        self.aug_prob = aug_prob
    def __len__(self): return len(self.stems)
    def __getitem__(self, i):
        base = self.stems[i]
        X = np.load(base + "_windows.npy", mmap_mode='r')   # (L,1,64,64) float16/32
        X = np.array(X, dtype=np.float32)
        S = np.load(base + "_winspans.npy")                 # (L,2)
        if self.with_labels:
            Y = np.load(base + "_winlabels.npy", mmap_mode='r')  # (L,4)
            Y = np.array(Y, dtype=np.float32)
        # to torch
        X = torch.from_numpy(X)
        if self.augment:

            for j in range(X.shape[0]):
                if torch.rand(1).item() < self.aug_prob:
                    X[j] = aug_window(X[j])
        if self.with_labels:
            return X, torch.from_numpy(Y), torch.from_numpy(S)
        return X, None, torch.from_numpy(S)

def pad_collate(batch):
    Xs, Ys, Ss = zip(*batch)
    Ls = [x.shape[0] for x in Xs]
    B = len(Xs); maxL = max(Ls)
    Xpad = torch.zeros(B, maxL, 1, 64, 64, dtype=torch.float32)
    mask = torch.zeros(B, maxL, dtype=torch.bool)
    Ypad = None if Ys[0] is None else torch.zeros(B, maxL, 4, dtype=torch.float32)
    for i,(X,Y) in enumerate(zip(Xs,Ys)):
        L = X.shape[0]
        Xpad[i,:L] = X
        mask[i,:L] = True
        if Y is not None:
            Ypad[i,:L] = Y
    return Xpad, Ypad, mask


def build_clip_sampler(stems):
    pres = []
    for base in stems:
        Y = np.load(base + "_winlabels.npy", mmap_mode='r')
        pres.append((Y.sum(axis=0) > 0).astype(int))   #
    pres = np.stack(pres, 0)               # (Nclips, 4)
    freq = pres.sum(axis=0) + 1e-6         # clips containing each class
    inv  = 1.0 / freq
    w = (pres * inv.reshape(1,4)).sum(axis=1)
    w[w == 0] = 0.1 * inv.sum()
    return torch.DoubleTensor(w)


class WindowCNN(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),  # 64x64 -> 32x32
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.proj = nn.Linear(128, 128)
    def forward(self, x):                       # x: (B*L,1,64,64)
        h = self.net(x).squeeze(-1).squeeze(-1) # (B*L,128)
        return self.proj(h)                     # (B*L,128)

class BahdanauAttn(nn.Module):
    def __init__(self, dim, attn_dim=128):
        super().__init__()
        self.W = nn.Linear(dim, attn_dim)
        self.u = nn.Linear(attn_dim, 1, bias=False)
    def forward(self, H, mask):                 # H: (B,L,D), mask: (B,L)
        A = torch.tanh(self.W(H))               # (B,L,A)
        e = self.u(A).squeeze(-1)               # (B,L)
        e = e.masked_fill(~mask, torch.finfo(e.dtype).min)  # safe mask
        alpha = torch.softmax(e, dim=1)         # (B,L)
        C = (H * alpha.unsqueeze(-1)).sum(1)    # (B,D)
        return C, alpha

class CNN_BiGRU_Attn(nn.Module):
    def __init__(self, emb_dim=128, rnn_hidden=128, num_classes=4):
        super().__init__()
        self.cnn = WindowCNN(emb_dim)
        self.rnn = nn.GRU(emb_dim, rnn_hidden, batch_first=True, bidirectional=True)
        self.attn = BahdanauAttn(2*rnn_hidden)
        self.cls_win  = nn.Linear(2*rnn_hidden, num_classes)  # per-window
        self.cls_clip = nn.Linear(2*rnn_hidden, num_classes)  # clip pooled
    def forward(self, X, mask):
        B,L,_,_,_ = X.shape
        Z = self.cnn(X.view(B*L,1,64,64)).view(B,L,-1)  # (B,L,128)
        Z = Z.masked_fill(~mask.unsqueeze(-1), 0)
        H, _ = self.rnn(Z)                              # (B,L,2*h)
        logits_win = self.cls_win(H)                    # (B,L,4)
        C, alpha = self.attn(H, mask)                   # (B,2*h),(B,L)
        logits_clip = self.cls_clip(C)                  # (B,4)
        return logits_win, logits_clip, alpha


class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=4.0, clip=0.05, eps=1e-8, reduction='mean'):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.clip = clip
        self.eps = eps
        self.reduction = reduction

    def forward(self, logits, targets):
        # supports shapes (..., C); works for (B,L,C) too
        x_sigmoid = torch.sigmoid(logits)
        xs_pos = x_sigmoid
        xs_neg = 1.0 - x_sigmoid

        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        los_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))

        if self.gamma_pos > 0 or self.gamma_neg > 0:
            pt_pos = xs_pos * targets
            pt_neg = xs_pos * (1.0 - targets)
            one_sided_w = (1.0 - pt_pos).pow(self.gamma_pos) * targets + \
                          (pt_neg).pow(self.gamma_neg) * (1.0 - targets)
            los_pos = los_pos * one_sided_w
            los_neg = los_neg * one_sided_w

        loss = -(los_pos + los_neg)
        return loss.mean() if self.reduction == 'mean' else loss


all_stems = list_stems(TRAIN_FEATURES_DIR)
train_stems, val_stems = split_train_val(all_stems, VAL_SPLIT, SEED)
print(f"[INFO] Clips: train={len(train_stems)} | val={len(val_stems)}")

train_ds = HFLungWindows(train_stems, with_labels=True, augment=AUG_TRAIN, aug_prob=AUG_PROB)
val_ds   = HFLungWindows(val_stems,   with_labels=True, augment=False)
test_ds  = HFLungWindows(list_stems(TEST_FEATURES_DIR), with_labels=True, augment=False)


train_weights = build_clip_sampler(train_stems)
train_sampler = WeightedRandomSampler(train_weights,
                                      num_samples=len(train_stems),
                                      replacement=True)

train_dl = DataLoader(train_ds, batch_size=BATCH_CLIPS, sampler=train_sampler,
                      collate_fn=pad_collate, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=(NUM_WORKERS>0), drop_last=False)

val_dl   = DataLoader(val_ds,   batch_size=BATCH_CLIPS, shuffle=False,
                      collate_fn=pad_collate, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=(NUM_WORKERS>0), drop_last=False)

test_dl  = DataLoader(test_ds,  batch_size=BATCH_CLIPS, shuffle=False,
                      collate_fn=pad_collate, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=(NUM_WORKERS>0), drop_last=False)


model = CNN_BiGRU_Attn().to(device)


if os.path.exists(SSL_ENCODER_PATH):
    state = torch.load(SSL_ENCODER_PATH, map_location=device)
    missing, unexpected = model.cnn.load_state_dict(state, strict=False)
    print("[INFO] Loaded SSL encoder. Missing:", missing, "| Unexpected:", unexpected)
else:
    print("[WARN] SSL encoder not found, training CNN from scratch.")


for p in model.cnn.parameters(): p.requires_grad = False


optim = torch.optim.AdamW([
    {"params": [p for n,p in model.named_parameters() if not n.startswith("cnn.")], "lr": BASE_LR_HEAD},
    {"params": [p for n,p in model.named_parameters() if n.startswith("cnn.")], "lr": 0.0},
], weight_decay=WEIGHT_DECAY)

def set_lrs(epoch):
    head_lr = BASE_LR_HEAD * (0.5 * (1 + np.cos(np.pi * epoch / max(1,EPOCHS))))
    cnn_lr  = (0.0 if epoch <= FREEZE_EPOCHS else BASE_LR_CNN * (0.5 * (1 + np.cos(np.pi * epoch / max(1,EPOCHS)))))
    optim.param_groups[0]['lr'] = head_lr
    optim.param_groups[1]['lr'] = cnn_lr

scaler = GradScaler()

# losses (ASL)
criterion_win  = AsymmetricLoss(gamma_pos=ASL_GAMMA_POS, gamma_neg=ASL_GAMMA_NEG, clip=ASL_CLIP, reduction='none')
criterion_clip = AsymmetricLoss(gamma_pos=ASL_GAMMA_POS, gamma_neg=ASL_GAMMA_NEG, clip=ASL_CLIP, reduction='mean')

# --------------------
# TRAIN / EVAL
# --------------------
def train_one_epoch(ep):
    model.train(); set_lrs(ep)
    run_loss = 0.0
    for X, Y, mask in tqdm(train_dl, desc=f"Train {ep}/{EPOCHS}"):
        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            logits_win, logits_clip, _ = model(X, mask)


            loss_mask = mask.unsqueeze(-1).float()                 # (B,L,1)
            loss_win_elem = criterion_win(logits_win, Y)           # (B,L,4)
            loss_win = (loss_win_elem * loss_mask).sum() / loss_mask.sum().clamp_min(1.0)


            clip_target = (Y.max(dim=1).values)                    # (B,4)
            loss_clip = criterion_clip(logits_clip, clip_target)

            loss = 0.7*loss_win + 0.3*loss_clip

        optim.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optim); scaler.update()
        run_loss += loss.item()
    return run_loss / max(1, len(train_dl))

@torch.no_grad()
def flatten_valid(Y, P, M):
    valid = M.reshape(-1) == 1
    y = Y.reshape(-1, 4)[valid]
    p = P.reshape(-1, 4)[valid]
    return y, p

@torch.no_grad()
def evaluate(loader):
    model.eval()
    Ys, Ps, Ms = [], [], []
    for X, Y, mask in tqdm(loader, desc="Eval"):
        X = X.to(device); mask = mask.to(device)
        logits_win, _, _ = model(X, mask)
        Ps.append(torch.sigmoid(logits_win).cpu().numpy())
        Ys.append(Y.numpy())
        Ms.append(mask.cpu().numpy())
    Y = np.concatenate(Ys, 0); P = np.concatenate(Ps, 0); M = np.concatenate(Ms, 0)
    y, p = flatten_valid(Y,P,M)
    yhat = (p >= 0.5).astype(np.int32)
    per_f1 = [f1_score(y[:,c], yhat[:,c], zero_division=0) for c in range(4)]
    # CAS = W ∨ R ∨ S (columns 0,2,3)
    y_cas = np.maximum.reduce([y[:,0], y[:,2], y[:,3]])
    p_cas = np.maximum.reduce([p[:,0], p[:,2], p[:,3]])
    yhat_cas = (p_cas >= 0.5).astype(np.int32)
    f1_cas = f1_score(y_cas, yhat_cas, zero_division=0)
    return float(np.mean(per_f1)), per_f1, f1_cas, (Y,P,M)

best_f1 = -1.0
for ep in range(1, EPOCHS+1):
    if ep == FREEZE_EPOCHS + 1:
        for p in model.cnn.parameters(): p.requires_grad = True
        print(f"[INFO] Unfroze CNN at epoch {ep}.")
    tr_loss = train_one_epoch(ep)
    val_macro, val_per, val_cas, _ = evaluate(val_dl)
    print(f"[E{ep}] loss={tr_loss:.4f} | VAL macro-F1={val_macro:.4f} | per-class [W C R S]={np.round(val_per,4)} | CAS={val_cas:.4f}")
    if val_macro > best_f1:
        best_f1 = val_macro
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  ↳ saved BEST model to {BEST_MODEL_PATH}")
print("Done. Best VAL macro-F1:", round(best_f1,4))


@torch.no_grad()
def evaluate_collect(loader):
    model.eval()
    Ys, Ps, Ms = [], [], []
    for X, Y, mask in tqdm(loader, desc="Validate (collect)"):
        X = X.to(device); mask = mask.to(device)
        logits_win, _, _ = model(X, mask)
        Ps.append(torch.sigmoid(logits_win).cpu().numpy())
        Ys.append(Y.numpy())
        Ms.append(mask.cpu().numpy())
    return np.concatenate(Ys, 0), np.concatenate(Ps, 0), np.concatenate(Ms, 0)

def pick_thresholds_on_val(Y, P, M):
    y, p = flatten_valid(Y,P,M)
    thrs = np.zeros(4, dtype=np.float32)
    for c in range(4):
        best_f1, best_t = -1.0, 0.5
        for t in np.linspace(0.05, 0.95, 19):
            yhat = (p[:, c] >= t).astype(np.int32)
            f1 = f1_score(y[:, c], yhat, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thrs[c] = best_t
    return thrs

@torch.no_grad()
def evaluate_with_thresholds(loader, thrs):
    model.eval()
    Ys, Ps, Ms = [], [], []
    for X, Y, mask in tqdm(loader, desc="Evaluate (thresholded)"):
        X = X.to(device); mask = mask.to(device)
        logits_win, _, _ = model(X, mask)
        Ps.append(torch.sigmoid(logits_win).cpu().numpy())
        Ys.append(Y.numpy())
        Ms.append(mask.cpu().numpy())
    Y = np.concatenate(Ys, 0); P = np.concatenate(Ps, 0); M = np.concatenate(Ms, 0)
    y, p = flatten_valid(Y,P,M)
    yhat = (p >= thrs.reshape(1,4)).astype(np.int32)
    per_f1 = [f1_score(y[:,c], yhat[:,c], zero_division=0) for c in range(4)]
    macro_f1 = float(np.mean(per_f1))
    # CAS
    y_cas = np.maximum.reduce([y[:,0], y[:,2], y[:,3]])
    p_cas = np.maximum.reduce([p[:,0], p[:,2], p[:,3]])
    yhat_cas = (p_cas >= max(thrs[0], thrs[2], thrs[3])).astype(np.int32)
    f1_cas = f1_score(y_cas, yhat_cas, zero_division=0)
    return macro_f1, per_f1, f1_cas


if os.path.exists(BEST_MODEL_PATH):
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device), strict=False)
    model.to(device).eval()
    print("[INFO] Loaded BEST model for threshold selection / test eval.")


Y_val, P_val, M_val = evaluate_collect(val_dl)
thrs = pick_thresholds_on_val(Y_val, P_val, M_val)

thrs = np.maximum(thrs, np.array([0.0, 0.0, 0.0, 0.20], dtype=np.float32))
print("VAL thresholds [W C R S] =", np.round(thrs, 3))


test_macro, test_per, test_cas = evaluate_with_thresholds(test_dl, thrs)
print(f"[TEST] window macro-F1 = {test_macro:.4f} | per-class [W C R S] = {np.round(test_per,4)} | CAS={test_cas:.4f}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.1 which is incompatible.
[INFO] Clips: train=7029 | val=780
[INFO] Loaded SSL encoder. Missing: [] | Unexpected: []


Eval: 100%|██████████| 195/195 [00:08<00:00, 24.27it/s]


[E1] loss=0.6526 | VAL macro-F1=0.3485 | per-class [W C R S]=[0.4921 0.5629 0.2739 0.0652] | CAS=0.5656
  ↳ saved BEST model to /content/drive/MyDrive/Conformer/hf_lung_model_best.pt


Eval: 100%|██████████| 195/195 [00:08<00:00, 23.91it/s]


[E2] loss=0.6342 | VAL macro-F1=0.3529 | per-class [W C R S]=[0.4891 0.5692 0.2833 0.07  ] | CAS=0.5686
  ↳ saved BEST model to /content/drive/MyDrive/Conformer/hf_lung_model_best.pt


...

[E50] loss=0.1717 | VAL macro-F1=0.5121 | per-class [W C R S]=[0.6442 0.657  0.5083 0.2388] | CAS=0.7185
Done. Best VAL macro-F1: 0.5127
[INFO] Loaded BEST model for threshold selection / test eval.


Validate (collect): 100%|██████████| 195/195 [00:07<00:00, 25.65it/s]


VAL thresholds [W C R S] = [0.55 0.5  0.65 0.65]


Evaluate (thresholded): 100%|██████████| 489/489 [00:19<00:00, 25.73it/s]

[TEST] window macro-F1 = 0.2158 | per-class [W C R S] = [0.285  0.3623 0.1999 0.0159] | CAS=0.3862


In [ ]:

import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from tqdm import tqdm


BEST_MODEL_PATH = "/content/drive/MyDrive/Conformer/hf_lung_model_best.pt"


N_ENSEMBLE = 4
EARLY_STOP_PATIENCE = 7
MAX_EPOCHS = EPOCHS
ENSEMBLE_DIR = os.path.dirname(BEST_MODEL_PATH)
ENSEMBLE_PREFIX = os.path.splitext(os.path.basename(BEST_MODEL_PATH))[0]
os.makedirs(ENSEMBLE_DIR, exist_ok=True)

def make_single_worker_loaders():

    train_weights = build_clip_sampler(train_stems)
    train_sampler = WeightedRandomSampler(train_weights, num_samples=len(train_stems), replacement=True)

    train_dl_ens = DataLoader(
        train_ds, batch_size=BATCH_CLIPS, sampler=train_sampler,
        collate_fn=pad_collate, num_workers=0, pin_memory=False,
        persistent_workers=False, drop_last=False
    )
    val_dl_ens = DataLoader(
        val_ds, batch_size=BATCH_CLIPS, shuffle=False,
        collate_fn=pad_collate, num_workers=0, pin_memory=False,
        persistent_workers=False, drop_last=False
    )
    test_dl_ens = DataLoader(
        test_ds, batch_size=BATCH_CLIPS, shuffle=False,
        collate_fn=pad_collate, num_workers=0, pin_memory=False,
        persistent_workers=False, drop_last=False
    )
    return train_dl_ens, val_dl_ens, test_dl_ens

train_dl_ens, val_dl_ens, test_dl_ens = make_single_worker_loaders()


def set_seed_local(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def set_lrs_for(optim, epoch, max_epochs=MAX_EPOCHS):
    head_lr = BASE_LR_HEAD * (0.5 * (1 + np.cos(np.pi * epoch / max(1, max_epochs))))
    cnn_lr  = (0.0 if epoch <= FREEZE_EPOCHS else BASE_LR_CNN * (0.5 * (1 + np.cos(np.pi * epoch / max(1, max_epochs)))))
    optim.param_groups[0]['lr'] = head_lr
    optim.param_groups[1]['lr'] = cnn_lr

def safe_roc_auc(y_true_bin, y_prob):
    if y_true_bin.min() == y_true_bin.max():
        return np.nan
    try:
        return roc_auc_score(y_true_bin, y_prob)
    except Exception:
        return np.nan

@torch.no_grad()
def flatten_valid(Y, P, M):
    valid = M.reshape(-1) == 1
    y = Y.reshape(-1, 4)[valid]
    p = P.reshape(-1, 4)[valid]
    return y, p

def compute_all_metrics_from_probs(Y, P, M, thresholds=None):
    y, p = flatten_valid(Y, P, M)
    thr = np.array([0.5,0.5,0.5,0.5], dtype=np.float32) if thresholds is None else np.array(thresholds, dtype=np.float32)
    yhat = (p >= thr.reshape(1,4)).astype(np.int32)

    per_f1 = np.array([f1_score(y[:,c], yhat[:,c], zero_division=0) for c in range(4)], dtype=np.float32)
    macro_f1 = float(per_f1.mean())

    micro_f1 = float(f1_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))
    precision = float(precision_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))
    recall = float(recall_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))

    per_acc = []
    for c in range(4):
        yc = y[:,c]; yh = yhat[:,c]
        acc = float((yc == yh).mean()) if yc.size > 0 else np.nan
        per_acc.append(acc)
    per_acc = np.array(per_acc, dtype=np.float32)

    per_auc = np.array([safe_roc_auc(y[:,c], p[:,c]) for c in range(4)], dtype=np.float32)
    macro_auc = float(np.nanmean(per_auc))


    y_cas = np.maximum.reduce([y[:,0], y[:,2], y[:,3]])
    p_cas = np.maximum.reduce([p[:,0], p[:,2], p[:,3]])
    yhat_cas = (p_cas >= max(thr[0], thr[2], thr[3])).astype(np.int32)
    cas_f1 = float(f1_score(y_cas, yhat_cas, zero_division=0))

    return {
        "macro_f1": macro_f1,
        "per_f1": per_f1,
        "micro_f1": micro_f1,
        "precision": precision,
        "recall": recall,
        "per_acc": per_acc,
        "macro_auc": macro_auc,
        "per_auc": per_auc,
        "cas_f1": cas_f1
    }

def pretty_print_metrics(tag, metrics):
    mf1 = metrics["macro_f1"]; per_f1 = metrics["per_f1"]; cas = metrics["cas_f1"]
    mic = metrics["micro_f1"]; prec = metrics["precision"]; rec = metrics["recall"]
    acc = metrics["per_acc"]; aucm = metrics["macro_auc"]; aucp = metrics["per_auc"]
    print(f"[{tag}] Macro-F1={mf1:.4f} | Per-F1 [W C R S]={np.round(per_f1,4)} | CAS-F1={cas:.4f}")
    print(f"       Micro-F1={mic:.4f} | Precision={prec:.4f} | Recall={rec:.4f}")
    print(f"       Per-class Acc [W C R S]={np.round(acc,4)} | Macro-AUC={aucm:.4f} | Per-AUC={np.round(aucp,4)}")


def make_fresh_model_and_optim():
    m = CNN_BiGRU_Attn().to(device)


    if os.path.exists(SSL_ENCODER_PATH):
        try:
            state = torch.load(SSL_ENCODER_PATH, map_location=device)
            missing, unexpected = m.cnn.load_state_dict(state, strict=False)
            print("[ENS] SSL encoder loaded. Missing:", missing, "| Unexpected:", unexpected)
        except Exception as e:
            print("[ENS] Could not load SSL encoder (shape mismatch). Skipping. Err:", e)
    else:
        print("[ENS] SSL encoder not found. Training CNN from scratch.")


    for p in m.cnn.parameters(): p.requires_grad = False

    opt = torch.optim.AdamW([
        {"params": [p for n,p in m.named_parameters() if not n.startswith("cnn.")], "lr": BASE_LR_HEAD},
        {"params": [p for n,p in m.named_parameters() if n.startswith("cnn.")], "lr": 0.0},
    ], weight_decay=WEIGHT_DECAY)

    return m, opt

@torch.no_grad()
def _quick_val_macro_f1(model_local):
    model_local.eval()
    Ys, Ps, Ms = [], [], []
    for X, Y, mask in val_dl_ens:
        X = X.to(device); mask = mask.to(device)
        logits_win, _, _ = model_local(X, mask)
        Ps.append(torch.sigmoid(logits_win).cpu().numpy())
        Ys.append(Y.numpy()); Ms.append(mask.cpu().numpy())
    Y = np.concatenate(Ys, 0); P = np.concatenate(Ps, 0); M = np.concatenate(Ms, 0)
    y, p = flatten_valid(Y, P, M)
    yhat = (p >= 0.5).astype(np.int32)
    return float(np.mean([f1_score(y[:,c], yhat[:,c], zero_division=0) for c in range(4)]))

def train_one_member(member_seed, save_path):
    set_seed_local(member_seed)
    model_m, optim_m = make_fresh_model_and_optim()
    scaler_m = GradScaler()

    best_val = -1.0
    bad = 0

    for ep in range(1, MAX_EPOCHS+1):
        if ep == FREEZE_EPOCHS + 1:
            for p in model_m.cnn.parameters(): p.requires_grad = True
            print(f"[ENS seed={member_seed}] Unfroze CNN at epoch {ep}.")

        model_m.train(); set_lrs_for(optim_m, ep)
        run_loss = 0.0
        for X, Y, mask in tqdm(train_dl_ens, desc=f"[ENS seed={member_seed}] Train {ep}/{MAX_EPOCHS}", leave=False):
            X = X.to(device, non_blocking=True)
            Y = Y.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                logits_win, logits_clip, _ = model_m(X, mask)
                loss_mask = mask.unsqueeze(-1).float()
                loss_win_elem = criterion_win(logits_win, Y)
                loss_win = (loss_win_elem * loss_mask).sum() / loss_mask.sum().clamp_min(1.0)
                clip_target = (Y.max(dim=1).values)
                loss_clip = criterion_clip(logits_clip, clip_target)
                loss = 0.7*loss_win + 0.3*loss_clip
            optim_m.zero_grad(set_to_none=True)
            scaler_m.scale(loss).backward()
            nn.utils.clip_grad_norm_(model_m.parameters(), GRAD_CLIP)
            scaler_m.step(optim_m); scaler_m.update()
            run_loss += loss.item()

        val_macro = _quick_val_macro_f1(model_m)
        if val_macro > best_val:
            best_val = val_macro; bad = 0
            torch.save(model_m.state_dict(), save_path)
            print(f"[ENS seed={member_seed}] E{ep} macroF1={val_macro:.4f}  ↳ saved {save_path}")
        else:
            bad += 1
            print(f"[ENS seed={member_seed}] E{ep} macroF1={val_macro:.4f}  (no improve {bad}/{EARLY_STOP_PATIENCE})")
            if bad >= EARLY_STOP_PATIENCE:
                print(f"[ENS seed={member_seed}] Early stop at epoch {ep}. Best={best_val:.4f}")
                break

    model_m.load_state_dict(torch.load(save_path, map_location=device), strict=False)
    model_m.eval()
    return model_m

@torch.no_grad()
def collect_probs(model_list, loader):
    all_P = []
    ref_Y = ref_M = None
    for m in model_list:
        Ys, Ps, Ms = [], [], []
        for X, Y, mask in loader:
            X = X.to(device); mask = mask.to(device)
            logits_win, _, _ = m(X, mask)
            Ps.append(torch.sigmoid(logits_win).cpu().numpy())
            Ys.append(Y.numpy()); Ms.append(mask.cpu().numpy())
        Y = np.concatenate(Ys, 0); P = np.concatenate(Ps, 0); M = np.concatenate(Ms, 0)
        all_P.append(P)
        if ref_Y is None:
            ref_Y, ref_M = Y, M
        else:

            assert ref_Y.shape == Y.shape and ref_M.shape == M.shape, "Loader mismatch across models."
    P_avg = np.mean(np.stack(all_P, 0), axis=0)
    return ref_Y, P_avg, ref_M

def pick_thresholds_on_val_from_avg(Y, P, M):
    y, p = flatten_valid(Y, P, M)
    thrs = np.zeros(4, dtype=np.float32)
    for c in range(4):
        best, tbest = -1.0, 0.5
        for t in np.linspace(0.05, 0.95, 19):
            yhat = (p[:, c] >= t).astype(np.int32)
            f1 = f1_score(y[:, c], yhat, zero_division=0)
            if f1 > best: best, tbest = f1, t
        thrs[c] = tbest

    thrs = np.maximum(thrs, np.array([0.0, 0.0, 0.0, 0.20], dtype=np.float32))
    return thrs


members = []


if os.path.exists(BEST_MODEL_PATH):
    m0 = CNN_BiGRU_Attn().to(device)
    m0.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device), strict=False)
    m0.eval()
    members.append(m0)
    print(f"[ENS] Added existing best model: {BEST_MODEL_PATH}")


base_seed = SEED
for k in range(N_ENSEMBLE):
    ckpt = os.path.join(ENSEMBLE_DIR, f"{ENSEMBLE_PREFIX}_ens{k+1}.pt")
    seed_k = base_seed + 100*(k+1)
    m_k = train_one_member(seed_k, ckpt)
    members.append(m_k)

print(f"[ENS] Total members in ensemble = {len(members)}")

Yv, Pv_avg, Mv = collect_probs(members, val_dl_ens)
thr_ens = pick_thresholds_on_val_from_avg(Yv, Pv_avg, Mv)
print("Ensemble VAL thresholds [W C R S] =", np.round(thr_ens, 3))


val_metrics = compute_all_metrics_from_probs(Yv, Pv_avg, Mv, thresholds=thr_ens)
pretty_print_metrics("ENS VAL (avg probs, tuned thr)", val_metrics)


Yt, Pt_avg, Mt = collect_probs(members, test_dl_ens)
test_metrics = compute_all_metrics_from_probs(Yt, Pt_avg, Mt, thresholds=thr_ens)
pretty_print_metrics("ENS TEST", test_metrics)


[ENS] Added existing best model: /content/drive/MyDrive/Conformer/hf_lung_model_best.pt
[ENS] SSL encoder loaded. Missing: [] | Unexpected: []


[ENS seed=1437] E1 macroF1=0.3492  ↳ saved /content/drive/MyDrive/Conformer/hf_lung_model_best_ens1.pt


[ENS seed=1437] E2 macroF1=0.3637  ↳ saved /content/drive/MyDrive/Conformer/hf_lung_model_best_ens1.pt


[ENS seed=1437] E3 macroF1=0.3677  ↳ saved /content/drive/MyDrive/Conformer/hf_lung_model_best_ens1.pt
[ENS seed=1437] Unfroze CNN at epoch 4.


[ENS seed=1437] E4 macroF1=0.3668  (no improve 1/7)


...

[ENS seed=1737] E28 macroF1=0.5021  (no improve 3/7)


[ENS seed=1737] E29 macroF1=0.4854  (no improve 4/7)


[ENS seed=1737] E30 macroF1=0.4979  (no improve 5/7)


[ENS seed=1737] E31 macroF1=0.5187  (no improve 6/7)


[ENS seed=1737] E32 macroF1=0.5189  (no improve 7/7)
[ENS seed=1737] Early stop at epoch 32. Best=0.5300
[ENS] Total members in ensemble = 5
Ensemble VAL thresholds [W C R S] = [0.4  0.55 0.45 0.55]
[ENS VAL (avg probs, tuned thr)] Macro-F1=0.5789 | Per-F1 [W C R S]=[0.656  0.6761 0.5852 0.3983] | CAS-F1=0.7049
       Micro-F1=0.8386 | Precision=0.8386 | Recall=0.8386
       Per-class Acc [W C R S]=[0.7578 0.7317 0.8978 0.9671] | Macro-AUC=0.8237 | Per-AUC=[0.8245 0.8058 0.896  0.7684]
[ENS TEST] Macro-F1=0.2382 | Per-F1 [W C R S]=[0.3617 0.3199 0.2225 0.0488] | CAS-F1=0.3298
       Micro-F1=0.7782 | Precision=0.7782 | Recall=0.7782
       Per-class Acc [W C R S]=[0.6862 0.6077 0.8315 0.9872] | Macro-AUC=0.5953 | Per-AUC=[0.6379 0.592  0.6044 0.5469]


In [ ]:

!pip -q install scikit-learn==1.5.1 tqdm==4.66.4

# ---- CONFIG ----
TRAIN_FEATURES_DIR = "/content/drive/MyDrive/HF_Lung_V1/train/feature"
TEST_FEATURES_DIR  = "/content/drive/MyDrive/HF_Lung_V1/test/feature"
BEST_MODEL_PATH    = "/content/drive/MyDrive/Conformer/hf_lung_model_best.pt"

SEED = 1337
BATCH_CLIPS = 4
NUM_WORKERS = 2
AUG_PROB = 0.5


import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from tqdm import tqdm


def set_seed(s=SEED):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class WindowCNN(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),  # 64x64 -> 32x32
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.proj = nn.Linear(128, 128)
    def forward(self, x):                       # x: (B*L,1,64,64)
        h = self.net(x).squeeze(-1).squeeze(-1) # (B*L,128)
        return self.proj(h)                     # (B*L,128)

class BahdanauAttn(nn.Module):
    def __init__(self, dim, attn_dim=128):
        super().__init__()
        self.W = nn.Linear(dim, attn_dim)
        self.u = nn.Linear(attn_dim, 1, bias=False)
    def forward(self, H, mask):                 # H: (B,L,D), mask: (B,L)
        A = torch.tanh(self.W(H))               # (B,L,A)
        e = self.u(A).squeeze(-1)               # (B,L)
        e = e.masked_fill(~mask, torch.finfo(e.dtype).min)
        alpha = torch.softmax(e, dim=1)         # (B,L)
        C = (H * alpha.unsqueeze(-1)).sum(1)    # (B,D)
        return C, alpha

class CNN_BiGRU_Attn(nn.Module):
    def __init__(self, emb_dim=128, rnn_hidden=128, num_classes=4):
        super().__init__()
        self.cnn = WindowCNN(emb_dim)
        self.rnn = nn.GRU(emb_dim, rnn_hidden, batch_first=True, bidirectional=True)
        self.attn = BahdanauAttn(2*rnn_hidden)
        self.cls_win  = nn.Linear(2*rnn_hidden, num_classes)  # per-window
        self.cls_clip = nn.Linear(2*rnn_hidden, num_classes)  # clip pooled
    def forward(self, X, mask):
        B,L,_,_,_ = X.shape
        Z = self.cnn(X.view(B*L,1,64,64)).view(B,L,-1)  # (B,L,128)
        Z = Z.masked_fill(~mask.unsqueeze(-1), 0)
        H, _ = self.rnn(Z)                              # (B,L,2*h)
        logits_win = self.cls_win(H)                    # (B,L,4)
        C, alpha = self.attn(H, mask)                   # (B,2*h),(B,L)
        logits_clip = self.cls_clip(C)                  # (B,4)
        return logits_win, logits_clip, alpha


def list_stems(feat_dir):
    stems = []
    for f in os.listdir(feat_dir):
        if f.endswith("_windows.npy"):
            stems.append(os.path.join(feat_dir, f[:-12]))
    stems.sort()
    return stems

class HFLungWindows(Dataset):
    def __init__(self, stems, with_labels=True):
        self.stems = stems
        self.with_labels = with_labels
    def __len__(self): return len(self.stems)
    def __getitem__(self, i):
        base = self.stems[i]
        X = np.load(base + "_windows.npy", mmap_mode='r')   # (L,1,64,64)
        X = torch.from_numpy(np.array(X, dtype=np.float32))
        S = np.load(base + "_winspans.npy")                 # (L,2)
        if self.with_labels:
            Y = np.load(base + "_winlabels.npy", mmap_mode='r')  # (L,4)
            Y = torch.from_numpy(np.array(Y, dtype=np.float32))
            return X, Y, torch.from_numpy(S)
        else:
            return X, None, torch.from_numpy(S)

def pad_collate(batch):
    Xs, Ys, Ss = zip(*batch)
    Ls = [x.shape[0] for x in Xs]
    B, maxL = len(Xs), max(Ls)
    Xpad = torch.zeros(B, maxL, 1, 64, 64, dtype=torch.float32)
    mask = torch.zeros(B, maxL, dtype=torch.bool)
    Ypad = None if Ys[0] is None else torch.zeros(B, maxL, 4, dtype=torch.float32)
    for i,(X,Y) in enumerate(zip(Xs,Ys)):
        L = X.shape[0]; Xpad[i,:L] = X; mask[i,:L] = True
        if Y is not None: Ypad[i,:L] = Y
    return Xpad, Ypad, mask


need_build = any(n not in globals() for n in ["val_dl","test_dl"])
if need_build:
    val_stems  = list_stems(TRAIN_FEATURES_DIR)[:max(1,int(0.1*len(list_stems(TRAIN_FEATURES_DIR))))]  # dummy split if you didn’t persist
    test_stems = list_stems(TEST_FEATURES_DIR)
    val_ds  = HFLungWindows(val_stems,  with_labels=True)
    test_ds = HFLungWindows(test_stems, with_labels=True)
    val_dl  = DataLoader(val_ds,  batch_size=BATCH_CLIPS, shuffle=False,
                         collate_fn=pad_collate, num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=(NUM_WORKERS>0))
    test_dl = DataLoader(test_ds, batch_size=BATCH_CLIPS, shuffle=False,
                         collate_fn=pad_collate, num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=(NUM_WORKERS>0))

# ---- ROBUST CHECKPOINT LOADER (no retrain) ----
def load_model_from_ckpt(model_obj, ckpt_path, map_location=device, strict=False):
    assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"
    sd = torch.load(ckpt_path, map_location=map_location)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    target_sd = model_obj.state_dict()
    filtered, skipped = {}, []
    for k, v in sd.items():
        k2 = k.replace("module.", "")
        if k2 in target_sd and target_sd[k2].shape == v.shape:
            filtered[k2] = v
        else:
            skipped.append(k)
    missing, unexpected = model_obj.load_state_dict(filtered, strict=strict)
    print(f"[LOAD] OK={len(filtered)} | missing={len(missing)} | unexpected={len(unexpected)} | skipped={len(skipped)}")
    if skipped:
        print("       skipped (name/shape mismatch):", skipped[:6], "..." if len(skipped)>6 else "")
    return model_obj


@torch.no_grad()
def evaluate_collect(loader, model_for_eval):
    model_for_eval.eval()
    Ys, Ps, Ms = [], [], []
    for X, Y, mask in tqdm(loader, desc="Collect"):
        X = X.to(device); mask = mask.to(device)
        logits_win, _, _ = model_for_eval(X, mask)
        Ps.append(torch.sigmoid(logits_win).cpu().numpy())
        Ys.append(Y.numpy()); Ms.append(mask.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0), np.concatenate(Ms,0)

@torch.no_grad()
def flatten_valid(Y, P, M):
    valid = M.reshape(-1) == 1
    y = Y.reshape(-1, 4)[valid]
    p = P.reshape(-1, 4)[valid]
    return y, p

def pick_thresholds_on_val(Y, P, M):
    y, p = flatten_valid(Y,P,M)
    thrs = np.zeros(4, dtype=np.float32)
    for c in range(4):
        best, tbest = -1.0, 0.5
        for t in np.linspace(0.05, 0.95, 19):
            yhat = (p[:, c] >= t).astype(np.int32)
            f1 = f1_score(y[:, c], yhat, zero_division=0)
            if f1 > best: best, tbest = f1, t
        thrs[c] = tbest

    thrs = np.maximum(thrs, np.array([0.0, 0.0, 0.0, 0.20], dtype=np.float32))
    return thrs

def safe_roc_auc(y_true_bin, y_prob):
    if y_true_bin.min() == y_true_bin.max():
        return np.nan
    try:
        return roc_auc_score(y_true_bin, y_prob)
    except Exception:
        return np.nan

def compute_all_metrics_from_probs(Y, P, M, thresholds=None):
    y, p = flatten_valid(Y, P, M)
    thr = np.array([0.5,0.5,0.5,0.5], dtype=np.float32) if thresholds is None else np.array(thresholds, dtype=np.float32)
    yhat = (p >= thr.reshape(1,4)).astype(np.int32)

    per_f1 = np.array([f1_score(y[:,c], yhat[:,c], zero_division=0) for c in range(4)], dtype=np.float32)
    macro_f1 = float(per_f1.mean())
    micro_f1 = float(f1_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))
    precision = float(precision_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))
    recall = float(recall_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))

    per_acc = np.array([(y[:,c] == yhat[:,c]).mean() if y.shape[0] else np.nan for c in range(4)], dtype=np.float32)
    per_auc = np.array([safe_roc_auc(y[:,c], p[:,c]) for c in range(4)], dtype=np.float32)
    macro_auc = float(np.nanmean(per_auc))


    y_cas = np.maximum.reduce([y[:,0], y[:,2], y[:,3]])
    p_cas = np.maximum.reduce([p[:,0], p[:,2], p[:,3]])
    yhat_cas = (p_cas >= max(thr[0], thr[2], thr[3])).astype(np.int32)
    cas_f1 = float(f1_score(y_cas, yhat_cas, zero_division=0))

    return {
        "macro_f1": macro_f1,
        "per_f1": per_f1,
        "micro_f1": micro_f1,
        "precision": precision,
        "recall": recall,
        "per_acc": per_acc,
        "macro_auc": macro_auc,
        "per_auc": per_auc,
        "cas_f1": cas_f1
    }

def pretty_print_metrics(tag, metrics):
    mf1 = metrics["macro_f1"]; per_f1 = metrics["per_f1"]; cas = metrics["cas_f1"]
    mic = metrics["micro_f1"]; prec = metrics["precision"]; rec = metrics["recall"]
    acc = metrics["per_acc"]; aucm = metrics["macro_auc"]; aucp = metrics["per_auc"]
    print(f"[{tag}] Macro-F1={mf1:.4f} | Per-F1 [W C R S]={np.round(per_f1,4)} | CAS-F1={cas:.4f}")
    print(f"       Micro-F1={mic:.4f} | Precision={prec:.4f} | Recall={rec:.4f}")
    print(f"       Per-class Acc [W C R S]={np.round(acc,4)} | Macro-AUC={aucm:.4f} | Per-AUC={np.round(aucp,4)}")


assert os.path.exists(BEST_MODEL_PATH), f"BEST_MODEL_PATH not found: {BEST_MODEL_PATH}"
model_eval = CNN_BiGRU_Attn().to(device)
model_eval = load_model_from_ckpt(model_eval, BEST_MODEL_PATH, strict=False).to(device).eval()


Y_val, P_val, M_val = evaluate_collect(val_dl, model_eval)
thrs = pick_thresholds_on_val(Y_val, P_val, M_val)
print("VAL thresholds [W C R S] =", np.round(thrs, 3))
val_metrics = compute_all_metrics_from_probs(Y_val, P_val, M_val, thresholds=thrs)
pretty_print_metrics("BEST VAL (tuned thr)", val_metrics)


Y_test, P_test, M_test = evaluate_collect(test_dl, model_eval)
test_metrics = compute_all_metrics_from_probs(Y_test, P_test, M_test, thresholds=thrs)
pretty_print_metrics("BEST TEST", test_metrics)


try:
    import pandas as pd, json
    out_dir = os.path.dirname(BEST_MODEL_PATH)
    os.makedirs(out_dir, exist_ok=True)
    pd.DataFrame([
        {"split":"VAL",  **{k:(v.tolist() if hasattr(v,"tolist") else v) for k,v in val_metrics.items()},
         "thr_W":thrs[0],"thr_C":thrs[1],"thr_R":thrs[2],"thr_S":thrs[3]},
        {"split":"TEST", **{k:(v.tolist() if hasattr(v,"tolist") else v) for k,v in test_metrics.items()}}
    ]).to_csv(os.path.join(out_dir, "best_model_metrics.csv"), index=False)
    print("Saved CSV:", os.path.join(out_dir, "best_model_metrics.csv"))
except Exception as e:
    print("CSV save skipped:", e)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.1 which is incompatible.
[LOAD] OK=38 | missing=0 | unexpected=0 | skipped=0


Collect: 100%|██████████| 195/195 [06:05<00:00,  1.87s/it]


VAL thresholds [W C R S] = [0.6  0.6  0.75 0.95]
[BEST VAL (tuned thr)] Macro-F1=0.8275 | Per-F1 [W C R S]=[0.8531 0.7186 0.8478 0.8905] | CAS-F1=0.7523
       Micro-F1=0.8928 | Precision=0.8928 | Recall=0.8928
       Per-class Acc [W C R S]=[0.8736 0.7902 0.9137 0.9938] | Macro-AUC=0.9296 | Per-AUC=[0.9341 0.8683 0.9579 0.9583]


Collect: 100%|██████████| 489/489 [15:42<00:00,  1.93s/it]


[BEST TEST] Macro-F1=0.2056 | Per-F1 [W C R S]=[0.261  0.3605 0.1778 0.0231] | CAS-F1=0.1849
       Micro-F1=0.7955 | Precision=0.7955 | Recall=0.7955
       Per-class Acc [W C R S]=[0.7205 0.6329 0.8423 0.9861] | Macro-AUC=0.5653 | Per-AUC=[0.5961 0.6278 0.5563 0.481 ]
Saved CSV: /content/drive/MyDrive/Conformer/best_model_metrics.csv


In [ ]:

!pip -q install scikit-learn==1.5.1 tqdm==4.66.4

# ---- CONFIG ----
TRAIN_FEATURES_DIR = "/content/drive/MyDrive/HF_Lung_V1/train/feature"
TEST_FEATURES_DIR  = "/content/drive/MyDrive/HF_Lung_V1/test/feature"
BEST_MODEL_PATH    = "/content/drive/MyDrive/Conformer/hf_lung_model_best.pt"

SEED = 1337
BATCH_CLIPS = 4
NUM_WORKERS = 2
STRIDOR_THR_FLOOR = 0.20     # keep your stridor floor
CLIP_POOL_MODE = "max"       # "max" or "mean"

# ---- IMPORTS ----
import os, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from tqdm import tqdm

# ---- DEVICE/SEED ----
def set_seed(s=SEED):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class WindowCNN(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),  # 64x64 -> 32x32
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.proj = nn.Linear(128, 128)
    def forward(self, x):                       # x: (B*L,1,64,64)
        h = self.net(x).squeeze(-1).squeeze(-1) # (B*L,128)
        return self.proj(h)                     # (B*L,128)

class BahdanauAttn(nn.Module):
    def __init__(self, dim, attn_dim=128):
        super().__init__()
        self.W = nn.Linear(dim, attn_dim)
        self.u = nn.Linear(attn_dim, 1, bias=False)
    def forward(self, H, mask):                 # H: (B,L,D), mask: (B,L)
        A = torch.tanh(self.W(H))               # (B,L,A)
        e = self.u(A).squeeze(-1)               # (B,L)
        e = e.masked_fill(~mask, torch.finfo(e.dtype).min)
        alpha = torch.softmax(e, dim=1)         # (B,L)
        C = (H * alpha.unsqueeze(-1)).sum(1)    # (B,D)
        return C, alpha

class CNN_BiGRU_Attn(nn.Module):
    def __init__(self, emb_dim=128, rnn_hidden=128, num_classes=4):
        super().__init__()
        self.cnn = WindowCNN(emb_dim)
        self.rnn = nn.GRU(emb_dim, rnn_hidden, batch_first=True, bidirectional=True)
        self.attn = BahdanauAttn(2*rnn_hidden)
        self.cls_win  = nn.Linear(2*rnn_hidden, num_classes)  # per-window
        self.cls_clip = nn.Linear(2*rnn_hidden, num_classes)  # clip pooled
    def forward(self, X, mask):
        B,L,_,_,_ = X.shape
        Z = self.cnn(X.view(B*L,1,64,64)).view(B,L,-1)  # (B,L,128)
        Z = Z.masked_fill(~mask.unsqueeze(-1), 0)
        H, _ = self.rnn(Z)                              # (B,L,2*h)
        logits_win = self.cls_win(H)                    # (B,L,4)
        C, alpha = self.attn(H, mask)                   # (B,2*h),(B,L)
        logits_clip = self.cls_clip(C)                  # (B,4)
        return logits_win, logits_clip, alpha

def list_stems(feat_dir):
    stems = []
    for f in os.listdir(feat_dir):
        if f.endswith("_windows.npy"):
            stems.append(os.path.join(feat_dir, f[:-12]))  # drop "_windows.npy"
    stems.sort()
    return stems

class HFLungWindows(Dataset):
    def __init__(self, stems, with_labels=True):
        self.stems = stems
        self.with_labels = with_labels
    def __len__(self): return len(self.stems)
    def __getitem__(self, i):
        base = self.stems[i]
        X = np.load(base + "_windows.npy", mmap_mode='r')   # (L,1,64,64)
        X = torch.from_numpy(np.array(X, dtype=np.float32))
        S = np.load(base + "_winspans.npy")                 # (L,2)
        if self.with_labels:
            Y = np.load(base + "_winlabels.npy", mmap_mode='r')  # (L,4)
            Y = torch.from_numpy(np.array(Y, dtype=np.float32))
            return X, Y, torch.from_numpy(S)
        else:
            return X, None, torch.from_numpy(S)

def pad_collate(batch):
    Xs, Ys, Ss = zip(*batch)
    Ls = [x.shape[0] for x in Xs]
    B, maxL = len(Xs), max(Ls)
    Xpad = torch.zeros(B, maxL, 1, 64, 64, dtype=torch.float32)
    mask = torch.zeros(B, maxL, dtype=torch.bool)
    Ypad = None if Ys[0] is None else torch.zeros(B, maxL, 4, dtype=torch.float32)
    for i,(X,Y) in enumerate(zip(Xs,Ys)):
        L = X.shape[0]; Xpad[i,:L] = X; mask[i,:L] = True
        if Y is not None: Ypad[i,:L] = Y
    return Xpad, Ypad, mask

# Build loaders (eval)
val_stems  = list_stems(TRAIN_FEATURES_DIR)[:max(1,int(0.1*len(list_stems(TRAIN_FEATURES_DIR))))]  # if you didn't persist a split
test_stems = list_stems(TEST_FEATURES_DIR)
val_dl  = DataLoader(HFLungWindows(val_stems,  with_labels=True), batch_size=BATCH_CLIPS, shuffle=False,
                     collate_fn=pad_collate, num_workers=NUM_WORKERS,
                     pin_memory=True, persistent_workers=(NUM_WORKERS>0))
test_dl = DataLoader(HFLungWindows(test_stems, with_labels=True), batch_size=BATCH_CLIPS, shuffle=False,
                     collate_fn=pad_collate, num_workers=NUM_WORKERS,
                     pin_memory=True, persistent_workers=(NUM_WORKERS>0))


def load_model_from_ckpt(model_obj, ckpt_path, map_location=device, strict=False):
    assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"
    sd = torch.load(ckpt_path, map_location=map_location)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    target_sd = model_obj.state_dict()
    filtered, skipped = {}, []
    for k, v in sd.items():
        k2 = k.replace("module.", "")
        if k2 in target_sd and target_sd[k2].shape == v.shape:
            filtered[k2] = v
        else:
            skipped.append(k)
    missing, unexpected = model_obj.load_state_dict(filtered, strict=strict)
    print(f"[LOAD] OK={len(filtered)} | missing={len(missing)} | unexpected={len(unexpected)} | skipped={len(skipped)}")
    if skipped:
        print("       skipped (name/shape mismatch):", skipped[:6], "..." if len(skipped)>6 else "")
    return model_obj


@torch.no_grad()
def evaluate_collect(loader, model_for_eval):
    model_for_eval.eval()
    Ys, Ps, Ms = [], [], []
    for X, Y, mask in tqdm(loader, desc="Collect"):
        X = X.to(device); mask = mask.to(device)
        logits_win, _, _ = model_for_eval(X, mask)
        Ps.append(torch.sigmoid(logits_win).cpu().numpy())
        Ys.append(Y.numpy()); Ms.append(mask.cpu().numpy())
    return np.concatenate(Ys,0), np.concatenate(Ps,0), np.concatenate(Ms,0)

@torch.no_grad()
def flatten_valid(Y, P, M):
    valid = M.reshape(-1) == 1
    y = Y.reshape(-1, 4)[valid]
    p = P.reshape(-1, 4)[valid]
    return y, p

def pick_thresholds_on_val(Y, P, M):
    y, p = flatten_valid(Y,P,M)
    thrs = np.zeros(4, dtype=np.float32)
    for c in range(4):
        best, tbest = -1.0, 0.5
        for t in np.linspace(0.05, 0.95, 19):
            yhat = (p[:, c] >= t).astype(np.int32)
            f1 = f1_score(y[:, c], yhat, zero_division=0)
            if f1 > best: best, tbest = f1, t
        thrs[c] = tbest
    thrs = np.maximum(thrs, np.array([0.0, 0.0, 0.0, STRIDOR_THR_FLOOR], dtype=np.float32))
    return thrs

def safe_roc_auc(y_true_bin, y_prob):
    if y_true_bin.min() == y_true_bin.max():
        return np.nan
    try:
        return roc_auc_score(y_true_bin, y_prob)
    except Exception:
        return np.nan

def compute_all_metrics_from_probs(Y, P, M, thresholds=None):
    y, p = flatten_valid(Y, P, M)
    thr = np.array([0.5,0.5,0.5,0.5], dtype=np.float32) if thresholds is None else np.array(thresholds, dtype=np.float32)
    yhat = (p >= thr.reshape(1,4)).astype(np.int32)

    per_f1 = np.array([f1_score(y[:,c], yhat[:,c], zero_division=0) for c in range(4)], dtype=np.float32)
    macro_f1 = float(per_f1.mean())
    micro_f1 = float(f1_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))
    precision = float(precision_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))
    recall = float(recall_score(y.reshape(-1), yhat.reshape(-1), zero_division=0, average='micro'))

    per_acc = np.array([(y[:,c] == yhat[:,c]).mean() if y.shape[0] else np.nan for c in range(4)], dtype=np.float32)
    per_auc = np.array([safe_roc_auc(y[:,c], p[:,c]) for c in range(4)], dtype=np.float32)
    macro_auc = float(np.nanmean(per_auc))


    y_cas = np.maximum.reduce([y[:,0], y[:,2], y[:,3]])
    p_cas = np.maximum.reduce([p[:,0], p[:,2], p[:,3]])
    yhat_cas = (p_cas >= max(thr[0], thr[2], thr[3])).astype(np.int32)
    cas_f1 = float(f1_score(y_cas, yhat_cas, zero_division=0))

    return {
        "macro_f1": macro_f1,
        "per_f1": per_f1,
        "micro_f1": micro_f1,
        "precision": precision,
        "recall": recall,
        "per_acc": per_acc,
        "macro_auc": macro_auc,
        "per_auc": per_auc,
        "cas_f1": cas_f1
    }

def pretty_print_metrics(tag, metrics):
    mf1 = metrics["macro_f1"]; per_f1 = metrics["per_f1"]; cas = metrics["cas_f1"]
    mic = metrics["micro_f1"]; prec = metrics["precision"]; rec = metrics["recall"]
    acc = metrics["per_acc"]; aucm = metrics["macro_auc"]; aucp = metrics["per_auc"]
    print(f"[{tag}] Macro-F1={mf1:.4f} | Per-F1 [W C R S]={np.round(per_f1,4)} | CAS-F1={cas:.4f}")
    print(f"       Micro-F1={mic:.4f} | Precision={prec:.4f} | Recall={rec:.4f}")
    print(f"       Per-class Acc [W C R S]={np.round(acc,4)} | Macro-AUC={aucm:.4f} | Per-AUC={np.round(aucp,4)}")


assert os.path.exists(BEST_MODEL_PATH), f"BEST_MODEL_PATH not found: {BEST_MODEL_PATH}"
model_eval = CNN_BiGRU_Attn().to(device)
model_eval = load_model_from_ckpt(model_eval, BEST_MODEL_PATH, strict=False).to(device).eval()


Y_val,  P_val,  M_val  = evaluate_collect(val_dl,  model_eval)
Y_test, P_test, M_test = evaluate_collect(test_dl, model_eval)


def flatten_valid_only(Y, P, M):
    valid = M.reshape(-1) == 1
    return Y.reshape(-1,4)[valid], P.reshape(-1,4)[valid]
y_v, p_v = flatten_valid_only(Y_val, P_val, M_val)
y_t, p_t = flatten_valid_only(Y_test, P_test, M_test)
print("Prevalence per class [W C R S] (window-level):")
print("  VAL :", np.round(y_v.mean(axis=0), 4))
print("  TEST:", np.round(y_t.mean(axis=0), 4))


thrs = pick_thresholds_on_val(Y_val, P_val, M_val)
print("VAL thresholds [W C R S] =", np.round(thrs, 3))


val_metrics  = compute_all_metrics_from_probs(Y_val,  P_val,  M_val,  thresholds=thrs)
test_metrics = compute_all_metrics_from_probs(Y_test, P_test, M_test, thresholds=thrs)
pretty_print_metrics("BEST VAL (tuned thr)", val_metrics)
pretty_print_metrics("BEST TEST",            test_metrics)

y_t_bool    = (y_t > 0.5)
yhat_t_bool = (p_t >= thrs.reshape(1,4))
def confusion_counts_bool(y_bool, yhat_bool):
    out = []
    for c in range(4):
        yc = y_bool[:, c]; yh = yhat_bool[:, c]
        tp = int(np.logical_and(yc, yh).sum())
        fp = int(np.logical_and(~yc, yh).sum())
        fn = int(np.logical_and(yc, ~yh).sum())
        tn = int(np.logical_and(~yc, ~yh).sum())
        out.append((tp, fp, fn, tn))
    return np.array(out, dtype=np.int64)
conf = confusion_counts_bool(y_t_bool, yhat_t_bool)
print("Confusion per class [TP, FP, FN, TN] (TEST):")
for i, lab in enumerate(["W","C","R","S"]):
    print(f"  {lab}: {conf[i].tolist()}")


best_thr_t = np.zeros(4, dtype=np.float32)
best_f1_t  = np.zeros(4, dtype=np.float32)
for c in range(4):
    fbest, tbest = -1.0, 0.5
    for t in np.linspace(0.05, 0.95, 19):
        f1 = f1_score(y_t[:,c], (p_t[:,c] >= t).astype(np.int32), zero_division=0)
        if f1 > fbest:
            fbest, tbest = f1, t
    best_thr_t[c] = tbest; best_f1_t[c] = fbest
print("Oracle thresholds on TEST [W C R S] =", np.round(best_thr_t,3))
print("Oracle per-class F1 on TEST         =", np.round(best_f1_t,4), "| macro =", round(float(best_f1_t.mean()),4))

@torch.no_grad()
def collect_per_clip(loader, model_for_eval):
    model_for_eval.eval()
    all_probs, all_targets = [], []
    for X, Y, mask in tqdm(loader, desc="Collect (per-clip)"):
        X = X.to(device); mask = mask.to(device)
        logits_win, _, _ = model_for_eval(X, mask)
        P = torch.sigmoid(logits_win).cpu().numpy()   # (B,L,4)
        Yb = Y.numpy()                                # (B,L,4)
        Mb = mask.cpu().numpy().astype(bool)          # (B,L)
        for b in range(P.shape[0]):
            valid = Mb[b]
            all_probs.append(P[b,valid])              # (Lvalid,4)
            all_targets.append(Yb[b,valid])           # (Lvalid,4)
    return all_targets, all_probs

def clip_reduce(Y_list, P_list, mode=CLIP_POOL_MODE):
    y_clip, p_clip = [], []
    for Yw, Pw in zip(Y_list, P_list):
        y_any = (Yw.max(axis=0) > 0).astype(np.int32)      # clip ground-truth presence
        if mode == "max":   p_red = Pw.max(axis=0)
        elif mode == "mean":p_red = Pw.mean(axis=0)
        else: raise ValueError("mode must be 'max' or 'mean'")
        y_clip.append(y_any); p_clip.append(p_red)
    return np.stack(y_clip,0), np.stack(p_clip,0)

# Collect per-clip
Yv_list, Pv_list = collect_per_clip(val_dl,  model_eval)
Yt_list, Pt_list = collect_per_clip(test_dl, model_eval)
ycv, pcv = clip_reduce(Yv_list, Pv_list, CLIP_POOL_MODE)
yct, pct = clip_reduce(Yt_list, Pt_list, CLIP_POOL_MODE)

# Tune clip-level thresholds on VAL
best_thr_clip = np.zeros(4, dtype=np.float32)
for c in range(4):
    fbest, tbest = -1.0, 0.5
    for t in np.linspace(0.05, 0.95, 19):
        f1 = f1_score(ycv[:,c], (pcv[:,c] >= t).astype(np.int32), zero_division=0)
        if f1 > fbest: fbest, tbest = f1, t
    best_thr_clip[c] = tbest
best_thr_clip = np.maximum(best_thr_clip, np.array([0.0,0.0,0.0, STRIDOR_THR_FLOOR], dtype=np.float32))
print("Clip-level VAL thresholds [W C R S] =", np.round(best_thr_clip,3))

# Clip-level TEST metrics
yhat_clip_t = (pct >= best_thr_clip.reshape(1,4)).astype(np.int32)
per_f1_clip = np.array([f1_score(yct[:,c], yhat_clip_t[:,c], zero_division=0) for c in range(4)], dtype=np.float32)
macro_f1_clip = float(per_f1_clip.mean())
print(f"[CLIP TEST] Macro-F1={macro_f1_clip:.4f} | Per-F1 [W C R S]={np.round(per_f1_clip,4)}")

# Clip-level CAS
p_cas_clip_t = np.maximum.reduce([pct[:,0], pct[:,2], pct[:,3]])
y_cas_clip_t = np.maximum.reduce([yct[:,0], yct[:,2], yct[:,3]])
thr_cas_clip = float(max(best_thr_clip[0], best_thr_clip[2], best_thr_clip[3]))
f1_cas_clip  = f1_score(y_cas_clip_t, (p_cas_clip_t >= thr_cas_clip).astype(np.int32), zero_division=0)
print(f"[CLIP TEST] CAS-F1={f1_cas_clip:.4f} (thr={thr_cas_clip:.2f})")


try:
    import pandas as pd, json
    out_dir = os.path.dirname(BEST_MODEL_PATH); os.makedirs(out_dir, exist_ok=True)
    pd.DataFrame([
        {"split":"VAL",  **{k:(v.tolist() if hasattr(v,"tolist") else v) for k,v in compute_all_metrics_from_probs(Y_val,P_val,M_val,thrs).items()},
         "thr_W":thrs[0],"thr_C":thrs[1],"thr_R":thrs[2],"thr_S":thrs[3]},
        {"split":"TEST", **{k:(v.tolist() if hasattr(v,"tolist") else v) for k,v in compute_all_metrics_from_probs(Y_test,P_test,M_test,thrs).items()}},
        {"split":"CLIP_TEST", "macro_f1":macro_f1_clip, "per_f1":per_f1_clip.tolist(), "cas_f1":f1_cas_clip,
         "note":"clip-level metrics with "+CLIP_POOL_MODE+" pooling"}
    ]).to_csv(os.path.join(out_dir, "best_model_metrics_eval_only.csv"), index=False)
    print("Saved CSV:", os.path.join(out_dir, "best_model_metrics_eval_only.csv"))
except Exception as e:
    print("CSV save skipped:", e)



[LOAD] OK=38 | missing=0 | unexpected=0 | skipped=0


Collect: 100%|██████████| 489/489 [00:18<00:00, 26.54it/s]


Prevalence per class [W C R S] (window-level):
  VAL : [0.4359 0.3513 0.2833 0.0295]
  TEST: [0.2071 0.1881 0.1513 0.0061]
VAL thresholds [W C R S] = [0.6  0.6  0.75 0.95]
[BEST VAL (tuned thr)] Macro-F1=0.8275 | Per-F1 [W C R S]=[0.8531 0.7186 0.8478 0.8905] | CAS-F1=0.7523
       Micro-F1=0.8928 | Precision=0.8928 | Recall=0.8928
       Per-class Acc [W C R S]=[0.8736 0.7902 0.9137 0.9938] | Macro-AUC=0.9296 | Per-AUC=[0.9341 0.8683 0.9579 0.9583]
[BEST TEST] Macro-F1=0.2056 | Per-F1 [W C R S]=[0.261  0.3605 0.1778 0.0231] | CAS-F1=0.1849
       Micro-F1=0.7955 | Precision=0.7955 | Recall=0.7955
       Per-class Acc [W C R S]=[0.7205 0.6329 0.8423 0.9861] | Macro-AUC=0.5653 | Per-AUC=[0.5961 0.6278 0.5563 0.481 ]
Confusion per class [TP, FP, FN, TN] (TEST):
  W: [2703, 6672, 8637, 36756]
  C: [5667, 15468, 4637, 28996]
  R: [934, 1283, 7354, 45197]
  S: [9, 433, 327, 53999]
Oracle thresholds on TEST [W C R S] = [0.25 0.55 0.15 0.95]
Oracle per-class F1 on TEST         = [0.3615 0.362

Collect (per-clip): 100%|██████████| 489/489 [00:18<00:00, 26.48it/s]


Clip-level VAL thresholds [W C R S] = [0.85 0.75 0.95 0.95]
[CLIP TEST] Macro-F1=0.2028 | Per-F1 [W C R S]=[0.2436 0.3707 0.168  0.029 ]
[CLIP TEST] CAS-F1=0.3629 (thr=0.95)
Saved CSV: /content/drive/MyDrive/Conformer/best_model_metrics_eval_only.csv
